# [실습 02] CoALA의 기억 3종(일화·의미·절차) 체험하기

> **연계**: 제1부 02장(CoALA) · **환경**: Google Colab · **모델**: 오픈웨이트 임베딩 `jhgan/ko-sroberta-multitask` (Hugging Face)

**학습 목표**
- CoALA의 장기기억 3종(일화·의미·절차)을 파이썬 자료구조로 표현한다.
- 오픈웨이트 임베딩으로 **의미기억 검색**(RAG의 핵심)을 직접 구현한다.

In [ ]:
!pip install -q sentence-transformers

## 1. 세 가지 장기기억을 자료구조로 표현

- **일화기억**: 과거 경험(대화 로그)
- **의미기억**: 사실·지식(문서)
- **절차기억**: 방법·기술(프롬프트·코드)

In [ ]:
episodic = ["어제 사용자는 환불 문의를 했고 성공적으로 처리됨"]
semantic = [
    "환불은 구매 후 7일 이내에 가능하다.",
    "배송비는 3만원 이상 구매 시 무료다.",
    "회원 등급은 실버, 골드, VIP 세 단계다.",
]
procedural = {"환불_처리": "1) 주문번호 확인 2) 기간 확인 3) 환불 실행"}
print("의미기억 문서 수:", len(semantic))

## 2. 의미기억 검색 (임베딩 + 유사도) — RAG의 핵심

오픈웨이트 임베딩 모델로 문서를 벡터화하고, 질문과 의미가 가까운 문서를 찾습니다. (03-2 절)

In [ ]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("jhgan/ko-sroberta-multitask")
doc_emb = embedder.encode(semantic, convert_to_tensor=True)

def retrieve(query, k=1):
    q = embedder.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q, doc_emb)[0]
    top = scores.argsort(descending=True)[:k]
    return [(semantic[i], float(scores[i])) for i in top]

print(retrieve("환불 언제까지 돼요?"))
print(retrieve("무료 배송 조건은?"))

## 3. 세 기억을 함께 사용하는 미니 에이전트

절차기억(방법) + 의미기억(검색된 사실) + 일화기억(과거 경험)을 한 번에 조합합니다.

In [ ]:
def answer(query):
    fact, score = retrieve(query, k=1)[0]
    return (
        f"[의미기억] 근거: {fact} (유사도 {score:.2f})\n"
        f"[일화기억] 참고 경험: {episodic[0]}\n"
        f"[절차기억] 처리 방법: {procedural['환불_처리']}"
    )

print(answer("환불하고 싶어요"))

## 4. 정리

- CoALA의 **일화·의미·절차** 기억을 자료구조로 구현했다.
- 오픈웨이트 임베딩으로 **의미기억 검색**(RAG)을 직접 만들었다.
- **더 해보기**: `semantic` 문서를 늘리고 `retrieve(k=2)`로 상위 2개를 함께 반환해 보세요.